# nb05 — Embedding Model Evaluation

Pick a production retriever for the Kalisio RAG pipeline. Corpus is EN-only,
most queries come in FR, so cross-lingual retrieval has to be measured
empirically.

| Layer       | Query style                  | Tests                         |
|-------------|-------------------------------|-------------------------------|
| `A_symbol`  | API / component name          | exact identifier match        |
| `B_docs`    | Natural-language question     | semantic understanding        |
| `C_code`    | Concept → file                | cross-modal (NL → code)       |
| `negative`  | Out-of-scope question         | rejection                     |

Gold comes from `outputs/nb05_gold.json` (authored in `experiments/nb05_embedding_eval/gold_draft.json`).

In [1]:
import os, sys, gc, json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 160)

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "outputs" / "nb05_gold.json").exists():
            return candidate
        knowledge = candidate / "knowledge"
        if (knowledge / "pyproject.toml").exists() and (knowledge / "outputs" / "nb05_gold.json").exists():
            return knowledge
    raise FileNotFoundError("Could not find knowledge project root with outputs/nb05_gold.json")

ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "experiments" / "nb05_embedding_eval"))

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[setup] device={DEVICE}", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

OUTPUTS = ROOT / "outputs"
OUTPUTS.mkdir(exist_ok=True)
GOLD_PATH = OUTPUTS / "nb05_gold.json"


[setup] device=cuda NVIDIA GeForce RTX 3060 Ti


## 1. Load corpus and gold

In [2]:
from corpus_filter import scan_corpus
from corpus_filter.models import FilterConfig
from corpus_filter.profiles import build_js_vue_rag_profile
from chunking import chunk_files
from nb05_helpers import load_gold_validated, gold_summary

_base = build_js_vue_rag_profile()
cfg = FilterConfig(
    excluded_dirs=_base.excluded_dirs - {"docs", "tools"},
    excluded_extensions=_base.excluded_extensions,
    excluded_filenames=_base.excluded_filenames,
    excluded_patterns=_base.excluded_patterns,
    max_file_size=_base.max_file_size,
    max_line_length=_base.max_line_length,
    included_extensions={".md", ".js", ".mjs", ".vue", ".json"},
)
scan = scan_corpus(config=cfg)
chunks = chunk_files(scan.included)
chunk_texts   = [c["text"] for c in chunks]
chunk_sources = [c["metadata"]["source"] for c in chunks]
print(f"[corpus] files={len(scan.included)}  chunks={len(chunks)}")

queries = load_gold_validated(GOLD_PATH, chunk_sources)
query_en = [q.en for q in queries]
query_fr = [q.fr for q in queries]
print(f"[gold]   {gold_summary(queries)}")


[corpus] files=1251  chunks=9893
[gold]   {'A_symbol': 60, 'B_docs': 80, 'C_code': 45, 'negative': 15, 'total': 200}


In [3]:
print("Sample query per layer:\n")
seen = set()
for q in queries:
    if q.layer in seen:
        continue
    seen.add(q.layer)
    print(f"[{q.layer}] {q.id}")
    print(f"  EN: {q.en}")
    print(f"  FR: {q.fr}")
    print(f"  gold: {list(q.gold_sources)}\n")


Sample query per layer:

[A_symbol] A-001
  EN: addLayer function
  FR: fonction addLayer
  gold: ['kdk/docs/api/map/map-mixins.md', 'kdk/docs/api/map/globe-mixins.md']

[B_docs] B-001
  EN: How do I add a new layer to a map?
  FR: Comment ajouter une nouvelle couche à la carte ?
  gold: ['kdk/docs/api/map/map-mixins.md', 'kdk/docs/api/map/globe-mixins.md']

[C_code] C-001
  EN: Where is the addLayer logic implemented for the 2D map?
  FR: Où est implémentée la logique addLayer pour la carte 2D ?
  gold: ['kdk/core/client/mixins/mixin.service.js', 'kdk/map/client/mixins/map/mixin.base-map.js']

[negative] N-001
  EN: How do I integrate TensorFlow.js for ML predictions?
  FR: Comment intégrer TensorFlow.js pour des prédictions ML ?
  gold: []



## 2. Candidate recipes and truncation audit

In [4]:
from nb05_helpers import RECIPES, truncation_audit

display(pd.DataFrame([
    {
        "key": k,
        "model_id": r.model_id,
        "query_prefix": repr(r.query_prefix),
        "passage_prefix": repr(r.passage_prefix),
        "max_tokens": r.max_tokens,
        "matryoshka_dim": r.matryoshka_dim,
        "family": r.family,
    } for k, r in RECIPES.items()
]).set_index("key"))

audit_rows = [truncation_audit(r, chunk_texts) for r in RECIPES.values()]
df_audit = pd.DataFrame(audit_rows).set_index("model")
df_audit


,model_id,query_prefix,passage_prefix,max_tokens,matryoshka_dim,family
key,,,,,,
bge-m3,BAAI/bge-m3,'','',8192,NaN,multilingual-dense
e5-large,intfloat/multilingual-e5-large,'query: ','passage: ',512,NaN,multilingual-dense
nomic-v1.5,nomic-ai/nomic-embed-text-v1.5,'search_query: ','search_document: ',8192,768.0,english-dense
jina-code,jinaai/jina-embeddings-v2-base-code,'','',8192,NaN,code-dense
e5-large-instruct,intfloat/multilingual-e5-large-instruct,"'Instruct: Given a developer question, retrieve the relevant Kalisio documentation page or sourc...",'',512,NaN,multilingual-dense
arctic-l-v2,Snowflake/snowflake-arctic-embed-l-v2.0,'query: ','',8192,NaN,multilingual-dense
qwen3-0.6b,Qwen/Qwen3-Embedding-0.6B,"'Instruct: Given a developer question in French or English, retrieve the relevant Kalisio docume...",'',8192,NaN,multilingual-code-dense


Token indices sequence length is longer than the specified maximum sequence length for this model (769 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (767 > 512). Running this sequence through the model will result in indexing errors


,max_tokens,n_chunks,n_truncated,truncation_rate,lost_fraction
model,,,,,
bge-m3,8192,9893,0,0.000000,0.000000
e5-large,512,9893,65,0.006570,0.010318
nomic-v1.5,8192,9893,0,0.000000,0.000000
jina-code,8192,9893,0,0.000000,0.000000
e5-large-instruct,512,9893,64,0.006469,0.010358
arctic-l-v2,8192,9893,0,0.000000,0.000000
qwen3-0.6b,8192,9893,0,0.000000,0.000000


## 3. Dense bake-off

In [5]:
from nb05_helpers import (
    encode_corpus, encode_queries, load_recipe_model,
    dense_rank, evaluate_ranks,
)

dense_ranks: dict[str, tuple[np.ndarray, np.ndarray]] = {}
dense_dfs: list[pd.DataFrame] = []
available_models: list[str] = []
skipped_models: list[dict[str, str]] = []

for key, recipe in RECIPES.items():
    print(f"[dense] {key} -> loading")
    model = None
    try:
        model = load_recipe_model(recipe)
        print(f"[dense] {key} -> encoding {len(chunks)} chunks")
        cv = encode_corpus(model, recipe, chunk_texts)
        en_v = encode_queries(model, recipe, query_en)
        fr_v = encode_queries(model, recipe, query_fr)

        en_r = dense_rank(en_v, cv)
        fr_r = dense_rank(fr_v, cv)
        dense_ranks[key] = (en_r, fr_r)
        available_models.append(key)

        dense_dfs.append(evaluate_ranks(en_r, queries, chunk_sources, language="en", approach=key))
        dense_dfs.append(evaluate_ranks(fr_r, queries, chunk_sources, language="fr", approach=key))
    except Exception as exc:
        skipped_models.append({"model": key, "error": type(exc).__name__, "message": str(exc)})
        print(f"[dense] SKIP {key}: {type(exc).__name__}: {exc}")
    finally:
        for var_name in ("model", "cv", "en_v", "fr_v"):
            if var_name in locals():
                del locals()[var_name]
        gc.collect()
        if torch.cuda.is_available():
            try:
                torch.cuda.empty_cache()
            except Exception:
                pass

if not dense_dfs:
    raise RuntimeError("No dense embedding model completed successfully.")

df_dense = pd.concat(dense_dfs, ignore_index=True)
print(f"\n[dense] available={available_models}")
if skipped_models:
    display(pd.DataFrame(skipped_models))
df_dense.head()


[dense] bge-m3 -> loading
[dense] bge-m3 -> encoding 9893 chunks
[dense] e5-large -> loading


XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[dense] e5-large -> encoding 9893 chunks
[dense] nomic-v1.5 -> loading


<All keys matched successfully>


[dense] nomic-v1.5 -> encoding 9893 chunks
[dense] jina-code -> loading
[dense] SKIP jina-code: ImportError: cannot import name 'find_pruneable_heads_and_indices' from 'transformers.pytorch_utils' (/home/felix/miniconda3/envs/knowledge/lib/python3.11/site-packages/transformers/pytorch_utils.py)
[dense] e5-large-instruct -> loading
[dense] e5-large-instruct -> encoding 9893 chunks
[dense] arctic-l-v2 -> loading
[dense] arctic-l-v2 -> encoding 9893 chunks
[dense] qwen3-0.6b -> loading
[dense] qwen3-0.6b -> encoding 9893 chunks

[dense] available=['bge-m3', 'e5-large', 'nomic-v1.5', 'e5-large-instruct', 'arctic-l-v2', 'qwen3-0.6b']


,model,error,message
0,jina-code,ImportError,cannot import name 'find_pruneable_heads_and_indices' from 'transformers.pytorch_utils' (/home/f...


,approach,language,layer,query_id,is_negative,hit@k,recall@k,mrr,hit@1,recall@1,hit@5,recall@5,hit@10,recall@10
0,bge-m3,en,A_symbol,A-001,False,0,0.0,0.014925,0,0.0,0,0.0,0,0.0
1,bge-m3,en,A_symbol,A-002,False,0,0.0,0.016393,0,0.0,0,0.0,0,0.0
2,bge-m3,en,A_symbol,A-003,False,1,1.0,1.000000,1,1.0,1,1.0,1,1.0
3,bge-m3,en,A_symbol,A-004,False,0,0.0,0.008000,0,0.0,0,0.0,0,0.0
4,bge-m3,en,A_symbol,A-005,False,0,0.0,0.004739,0,0.0,0,0.0,0,0.0


## 4. BM25 baseline

In [6]:
from nb05_helpers import bm25_rank

bm25_en_ranks = bm25_rank(chunk_texts, query_en)
bm25_fr_ranks = bm25_rank(chunk_texts, query_fr)
df_bm25 = pd.concat([
    evaluate_ranks(bm25_en_ranks, queries, chunk_sources, language="en", approach="bm25"),
    evaluate_ranks(bm25_fr_ranks, queries, chunk_sources, language="fr", approach="bm25"),
], ignore_index=True)
df_bm25.groupby(["layer", "language"])["hit@k"].mean().unstack("language").round(3)


language,en,fr
layer,,
A_symbol,0.850,0.833
B_docs,0.400,0.100
C_code,0.311,0.089
negative,0.333,0.000


## 5. Hybrid (Dense + BM25 via RRF)

In [7]:
from nb05_helpers import rrf_fuse

if not available_models:
    print("[hybrid] no dense models available, skipping")
    df_hybrid = pd.DataFrame()
else:
    leader_key = (
        df_dense.groupby("approach")["hit@k"].mean().idxmax()
    )
    print(f"[hybrid] leader = {leader_key}")
    en_leader, fr_leader = dense_ranks[leader_key]

    hybrid_dfs = []
    for k_rrf in (30, 60, 90):
        en_fused = rrf_fuse(en_leader, bm25_en_ranks, k_rrf=k_rrf)
        fr_fused = rrf_fuse(fr_leader, bm25_fr_ranks, k_rrf=k_rrf)
        approach = f"hybrid_k{k_rrf}"
        hybrid_dfs.append(evaluate_ranks(en_fused, queries, chunk_sources, language="en", approach=approach))
        hybrid_dfs.append(evaluate_ranks(fr_fused, queries, chunk_sources, language="fr", approach=approach))
    df_hybrid = pd.concat(hybrid_dfs, ignore_index=True)

df_hybrid.groupby(["approach", "layer"])["hit@k"].mean().unstack("layer").round(3)


[hybrid] leader = qwen3-0.6b


layer,A_symbol,B_docs,C_code,negative
approach,,,,
hybrid_k30,0.867,0.600,0.533,0.033
hybrid_k60,0.875,0.575,0.589,0.033
hybrid_k90,0.875,0.556,0.578,0.033


## 6. Cross-encoder reranker (Layer B + C only)

In [8]:
from nb05_helpers import load_reranker, rerank_topk

K_CAND = 20
df_rerank = pd.DataFrame()
if df_hybrid.empty:
    print("[reranker] skipped (no hybrid baseline)")
else:
    try:
        reranker = load_reranker()
    except Exception as exc:
        print(f"[reranker] unavailable: {type(exc).__name__}: {exc}")
        reranker = None

    if reranker is not None:
        best_k = (
            df_hybrid.groupby("approach")["hit@k"].mean().idxmax().split("_k")[1]
        )
        best_k = int(best_k)
        print(f"[reranker] reranking on top of hybrid_k{best_k}")
        en_leader, fr_leader = dense_ranks[leader_key]
        en_fused = rrf_fuse(en_leader, bm25_en_ranks, k_rrf=best_k)
        fr_fused = rrf_fuse(fr_leader, bm25_fr_ranks, k_rrf=best_k)

        rerank_eligible = [i for i, q in enumerate(queries) if q.layer in {"B_docs", "C_code"}]

        def rerank_one(fused: np.ndarray, q_text: str, qi: int) -> np.ndarray:
            cand = fused[qi, :K_CAND].tolist()
            order = rerank_topk(reranker, q_text, [chunks[c]["text"] for c in cand])
            reranked = [cand[o] for o in order]
            return np.array(reranked + fused[qi, K_CAND:].tolist())

        en_reranked = en_fused.copy()
        fr_reranked = fr_fused.copy()
        for qi in rerank_eligible:
            en_reranked[qi] = rerank_one(en_fused, queries[qi].en, qi)
            fr_reranked[qi] = rerank_one(fr_fused, queries[qi].fr, qi)

        df_rerank = pd.concat([
            evaluate_ranks(en_reranked, queries, chunk_sources, language="en", approach=f"hybrid_k{best_k}+rerank"),
            evaluate_ranks(fr_reranked, queries, chunk_sources, language="fr", approach=f"hybrid_k{best_k}+rerank"),
        ], ignore_index=True)

        del reranker
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

if not df_rerank.empty:
    display(df_rerank[df_rerank["layer"].isin({"B_docs", "C_code"})]
            .groupby(["layer", "language"])["hit@k"].mean().unstack("language").round(3))


[reranker] reranking on top of hybrid_k60


language,en,fr
layer,,
B_docs,0.612,0.575
C_code,0.778,0.533


## 7. Per-layer leaderboard

In [9]:
from nb05_helpers import leaderboard

df_all = pd.concat([df_dense, df_bm25, df_hybrid, df_rerank], ignore_index=True)

for layer in ("A_symbol", "B_docs", "C_code", "negative"):
    print(f"\n=== {layer} ===")
    display(leaderboard(df_all, layer=layer))

# Negative stress view: hit@1/5/10 means rejection success for negative queries.
# If it drops as K grows, retrieval is pulling plausible-but-wrong Kalisio sources.
negative_cols = [c for c in ("hit@1", "hit@5", "hit@10") if c in df_all.columns]
df_negative_stress = (
    df_all[df_all["is_negative"]]
    .groupby(["approach", "language"])[negative_cols]
    .mean()
    .rename(columns={"hit@1": "reject@1", "hit@5": "reject@5", "hit@10": "reject@10"})
    .round(3)
    .sort_values(["language", "reject@5", "reject@1"], ascending=[True, False, False])
)
print("\n=== Negative stress: rejection success by K ===")
display(df_negative_stress)



=== A_symbol ===


language,en,fr,mean
approach,,,
e5-large,0.883,0.883,0.883
qwen3-0.6b,0.883,0.883,0.883
hybrid_k60+rerank,0.883,0.867,0.875
hybrid_k90,0.883,0.867,0.875
hybrid_k60,0.883,0.867,0.875
hybrid_k30,0.867,0.867,0.867
nomic-v1.5,0.867,0.817,0.842
bm25,0.850,0.833,0.841
arctic-l-v2,0.850,0.817,0.833



=== B_docs ===


language,en,fr,mean
approach,,,
qwen3-0.6b,0.725,0.688,0.706
arctic-l-v2,0.638,0.612,0.625
e5-large,0.625,0.575,0.600
hybrid_k30,0.638,0.562,0.600
hybrid_k60+rerank,0.612,0.575,0.593
hybrid_k60,0.650,0.500,0.575
bge-m3,0.588,0.525,0.556
hybrid_k90,0.650,0.462,0.556
e5-large-instruct,0.625,0.425,0.525



=== C_code ===


language,en,fr,mean
approach,,,
hybrid_k60+rerank,0.778,0.533,0.656
qwen3-0.6b,0.756,0.511,0.634
arctic-l-v2,0.667,0.556,0.612
e5-large,0.556,0.622,0.589
hybrid_k60,0.667,0.511,0.589
hybrid_k90,0.667,0.489,0.578
bge-m3,0.600,0.511,0.556
hybrid_k30,0.622,0.444,0.533
e5-large-instruct,0.467,0.400,0.434



=== negative ===


language,en,fr,mean
approach,,,
bm25,0.333,0.000,0.166
arctic-l-v2,0.067,0.200,0.134
bge-m3,0.133,0.133,0.133
e5-large,0.200,0.000,0.100
e5-large-instruct,0.133,0.000,0.066
nomic-v1.5,0.133,0.000,0.066
hybrid_k30,0.067,0.000,0.034
hybrid_k60,0.067,0.000,0.034
hybrid_k60+rerank,0.067,0.000,0.034



=== Negative stress: rejection success by K ===


,,reject@1,reject@5,reject@10
approach,language,,,
bm25,en,0.600,0.333,0.067
e5-large,en,0.467,0.200,0.133
nomic-v1.5,en,0.733,0.133,0.000
e5-large-instruct,en,0.533,0.133,0.000
bge-m3,en,0.400,0.133,0.000
arctic-l-v2,en,0.600,0.067,0.000
hybrid_k60,en,0.533,0.067,0.000
hybrid_k60+rerank,en,0.533,0.067,0.000
hybrid_k90,en,0.533,0.067,0.067


## 8. Cost profile

In [10]:
from nb05_helpers import measure_throughput, index_size_bytes, query_latency_ms

cost_rows = []
sample_q = query_en[: min(40, len(query_en))]
sample_chunks = chunk_texts[: min(512, len(chunk_texts))]

for key in available_models:
    recipe = RECIPES[key]
    try:
        model = load_recipe_model(recipe)
    except Exception:
        continue

    thr = measure_throughput(model, recipe, sample_chunks)
    full_cv = encode_corpus(model, recipe, chunk_texts)
    encode_q = lambda q, m=model, r=recipe: encode_queries(m, r, [q])
    lat = query_latency_ms(encode_q, full_cv, sample_q)

    cost_rows.append({
        "model": key,
        "dim": int(full_cv.shape[1]),
        "chunks_per_sec": round(thr["chunks_per_sec"], 1),
        "query_mean_ms": round(lat["mean_ms"], 1),
        "query_p95_ms":  round(lat["p95_ms"], 1),
        "index_mb": round(index_size_bytes(len(chunks), full_cv.shape[1]) / (1024 ** 2), 1),
    })
    del model, full_cv
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

df_cost = pd.DataFrame(cost_rows).set_index("model")
df_cost


XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
<All keys matched successfully>


,dim,chunks_per_sec,query_mean_ms,query_p95_ms,index_mb
model,,,,,
bge-m3,1024,53.1,11.0,11.2,38.6
e5-large,1024,51.3,11.0,11.2,38.6
nomic-v1.5,768,130.8,8.5,8.7,29.0
e5-large-instruct,1024,170.6,38.7,39.4,38.6
arctic-l-v2,1024,52.6,11.0,11.0,38.6
qwen3-0.6b,1024,91.0,23.7,25.1,38.6


## 9. Per-layer winners

In [11]:
from nb05_helpers import per_layer_summary

per_layer = per_layer_summary(df_all)
winners = {layer: per_layer[layer].idxmax() for layer in ("A_symbol", "B_docs", "C_code")}
print("Winners:", winners)
display(per_layer)


Winners: {'A_symbol': 'e5-large', 'B_docs': 'qwen3-0.6b', 'C_code': 'hybrid_k60+rerank'}


layer,A_symbol,B_docs,C_code,negative
approach,,,,
arctic-l-v2,0.833,0.625,0.611,0.133
bge-m3,0.808,0.556,0.556,0.133
bm25,0.842,0.250,0.200,0.167
e5-large,0.883,0.600,0.589,0.100
e5-large-instruct,0.833,0.525,0.433,0.067
hybrid_k30,0.867,0.600,0.533,0.033
hybrid_k60,0.875,0.575,0.589,0.033
hybrid_k60+rerank,0.875,0.594,0.656,0.033
hybrid_k90,0.875,0.556,0.578,0.033


## 10. Persist results

In [12]:
df_all.to_json(OUTPUTS / "nb05_results.json", orient="records", indent=2)
df_cost.to_json(OUTPUTS / "nb05_cost.json", orient="index", indent=2)
df_audit.to_json(OUTPUTS / "nb05_truncation.json", orient="index", indent=2)
if "df_negative_stress" in globals():
    df_negative_stress.to_json(OUTPUTS / "nb05_negative_stress.json", orient="index", indent=2)
print("[save] outputs:")
for p in ("nb05_results.json", "nb05_cost.json", "nb05_truncation.json", "nb05_negative_stress.json"):
    print(" -", OUTPUTS / p)


[save] outputs:
 - /home/felix/kalisio/knowledge/outputs/nb05_results.json
 - /home/felix/kalisio/knowledge/outputs/nb05_cost.json
 - /home/felix/kalisio/knowledge/outputs/nb05_truncation.json
 - /home/felix/kalisio/knowledge/outputs/nb05_negative_stress.json


## 11. FR-first view

In [13]:
# FR-only per-layer leaderboard
fr = df_all[df_all["language"] == "fr"]
fr_per_layer = (
    fr.groupby(["approach", "layer"])["hit@k"].mean()
    .unstack("layer").round(3)
)
# Order columns deterministically; layers may be missing if no queries
ordered_cols = [c for c in ("A_symbol", "B_docs", "C_code", "negative") if c in fr_per_layer.columns]
fr_per_layer = fr_per_layer[ordered_cols]
fr_per_layer["B+C mean"] = fr_per_layer[[c for c in ("B_docs", "C_code") if c in fr_per_layer.columns]].mean(axis=1).round(3)
fr_per_layer = fr_per_layer.sort_values("B+C mean", ascending=False)
print("=== FR-only hit@5 by approach × layer (main metric; hit@1/hit@10 are in nb05_results.json) ===\n")
display(fr_per_layer)


=== FR-only hit@5 by approach × layer (main metric; hit@1/hit@10 are in nb05_results.json) ===



layer,A_symbol,B_docs,C_code,negative,B+C mean
approach,,,,,
qwen3-0.6b,0.883,0.688,0.511,0.000,0.599
e5-large,0.883,0.575,0.622,0.000,0.598
arctic-l-v2,0.817,0.612,0.556,0.200,0.584
hybrid_k60+rerank,0.867,0.575,0.533,0.000,0.554
bge-m3,0.817,0.525,0.511,0.133,0.518
hybrid_k60,0.867,0.500,0.511,0.000,0.506
hybrid_k30,0.867,0.562,0.444,0.000,0.503
hybrid_k90,0.867,0.462,0.489,0.000,0.476
e5-large-instruct,0.817,0.425,0.400,0.000,0.412


In [14]:
# Cost-accuracy frontier on FR mean (B+C layers — where the hard work is)
if not df_cost.empty:
    fr_score = (
        df_all[(df_all["language"] == "fr") & (df_all["layer"].isin(["B_docs", "C_code"]))]
        .groupby("approach")["hit@k"].mean()
    )
    frontier = (
        df_cost.assign(fr_BC_hit5=df_cost.index.map(fr_score).round(3))
        .dropna(subset=["fr_BC_hit5"])
        .sort_values("fr_BC_hit5", ascending=False)
        [["dim", "chunks_per_sec", "query_mean_ms", "index_mb", "fr_BC_hit5"]]
    )
    print("=== Dense models: FR B+C hit@5 vs cost ===\n")
    display(frontier)
else:
    frontier = pd.DataFrame()
    print("(no cost data — dense models did not load)")


=== Dense models: FR B+C hit@5 vs cost ===



,dim,chunks_per_sec,query_mean_ms,index_mb,fr_BC_hit5
model,,,,,
qwen3-0.6b,1024,91.0,23.7,38.6,0.624
e5-large,1024,51.3,11.0,38.6,0.592
arctic-l-v2,1024,52.6,11.0,38.6,0.592
bge-m3,1024,53.1,11.0,38.6,0.520
e5-large-instruct,1024,170.6,38.7,38.6,0.416
nomic-v1.5,768,130.8,8.5,29.0,0.240


In [15]:
def _layer_mean(df, lang, layers):
    sub = df[(df["language"] == lang) & (df["layer"].isin(layers))]
    return sub.groupby("approach")["hit@k"].mean()

# FR-weighted score: 0.6 FR(B+C) + 0.2 FR(A) + 0.2 EN(B+C)
fr_bc  = _layer_mean(df_all, "fr", ["B_docs", "C_code"])
fr_a   = _layer_mean(df_all, "fr", ["A_symbol"])
en_bc  = _layer_mean(df_all, "en", ["B_docs", "C_code"])
neg_fr = _layer_mean(df_all, "fr", ["negative"])

scored = (0.6 * fr_bc + 0.2 * fr_a + 0.2 * en_bc).sort_values(ascending=False).round(3)
top_table = pd.DataFrame({
    "weighted_score": scored,
    "FR_B+C": fr_bc.round(3),
    "FR_A":   fr_a.round(3),
    "EN_B+C": en_bc.round(3),
    "FR_neg": neg_fr.round(3),
}).loc[scored.index]
display(top_table.head(10))
print(f"Top-3: {scored.head(3).index.tolist()}")


,weighted_score,FR_B+C,FR_A,EN_B+C,FR_neg
approach,,,,,
qwen3-0.6b,0.698,0.624,0.883,0.736,0.000
e5-large,0.652,0.592,0.883,0.600,0.000
arctic-l-v2,0.648,0.592,0.817,0.648,0.200
hybrid_k60+rerank,0.644,0.560,0.867,0.672,0.000
hybrid_k30,0.612,0.520,0.867,0.632,0.000
hybrid_k60,0.607,0.504,0.867,0.656,0.000
bge-m3,0.594,0.520,0.817,0.592,0.133
hybrid_k90,0.588,0.472,0.867,0.656,0.000
e5-large-instruct,0.527,0.416,0.817,0.568,0.000


Top-3: ['qwen3-0.6b', 'e5-large', 'arctic-l-v2']
